In [5]:
import pandas as pd
from dotenv import load_dotenv
import os
from typing import List, Tuple

from tqdm import tqdm

from concurrent.futures import ThreadPoolExecutor, as_completed

from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import SystemMessage, HumanMessage

In [6]:
df_final = pd.read_csv('sampled_questions - agg.csv')
df_final_labels = df_final.loc[:, ['question', 'final']]
df_final_labels.head()

,question,final
0,"Девушки, кто-нибудь подал на увеличение пособи...",1
1,"А вы первый раз подавали? Я повторно, просто",0
2,Что вы тогда переживаете? У вас справка на рук...,0
3,"Победа, можно личный вопрос на публику? Даже д...",0
4,"Доброе утро,скажите,пожалуйста,а есть у нас в ...",1


In [7]:
load_dotenv('/Users/egorsmirnov/Desktop/FAQ_autocollector/.env')
parser = StrOutputParser()

models = ['gpt-oss-20b', 'gpt-oss-120b', 'llama-4-scout', 'llama-4-maverick', 'qwen3-14b', 'qwen3-32b', 'qwen3-235b-a22b']

API_KEY = os.getenv('API_KEY')
url = 'https://bothub.chat/api/v2/openai/v1'

llm = ChatOpenAI(api_key=API_KEY, base_url=url, model='llama-4-maverick')

SYSTEM_PROMPT = """
Ты — классификатор вопросов. Твоя задача: определить, является ли вопрос ИНФОРМАТИВНЫМ или НЕИНФОРМАТИВНЫМ.

Верни строго одно число:
1 — если вопрос информативный
0 — если вопрос неинформативный

Определения:ИНФОРМАТИВНЫЙ ВОПРОС — это вопрос, который имеет явную цель получить новую, недостающую информацию. Человек хочет узнать факт, способ действия, местоположение, процедуру, контакт, цену, наличие, правила, инструкции, рекомендации или другую конкретную информацию.  
Признаки:
- вопрос направлен на восполнение знаний
- содержит запрос на факты, инструкции, опыт, советы
- человек действительно хочет что‑то узнать

НЕИНФОРМАТИВНЫЙ ВОПРОС — это вопрос, который НЕ направлен на получение новой информации.  
Признаки:
- риторический вопрос
- продолжение диалога без запроса знаний
- выражение эмоций, иронии, шутки
- общеизвестный факт, не требующий ответа
- вопрос, не предполагающий полезного ответа

Примеры ИНФОРМАТИВНЫХ вопросов:
- «Где купить свежие морепродукты рядом с Убудом?»
- «Как записаться на маникюр за 5300 ₽?»
- «Кто может помыть окна в ЖК? Поделитесь контактами.»
- «Где посмотреть кроватку ребёнку после колыбельки?»
- «Есть тут фотограф для семейной фотосессии?»

Примеры НЕИНФОРМАТИВНЫХ вопросов:
- «Я могу написать тебе в лс?»
- «А зачем это строить»
- «Бюрки где?» (если контекст отсутствует)
- «А под что маскировать?»
- «Рей или Каору?»
- «Вам таблетки или травы?»

Формат ответа:
Верни только 0 или 1. Никаких пояснений, текста или комментариев.
"""


messages = [SystemMessage(content=SYSTEM_PROMPT), HumanMessage(content=str(df_final_labels.iloc[0, 0]))]

llm_response = llm.invoke(messages)
result = parser.invoke(llm_response)


print(result)

1


In [10]:
class LLM_classifier:
    
    def __init__(self, model, API_KEY, SYSTEM_PROMPT, url):
        self.parser = StrOutputParser()
        self.model = model
        self.API_KEY = API_KEY
        self.SYSTEM_PROMPT = SYSTEM_PROMPT
        self.url = url
        
    def classify_question(self, llm, question) -> str:
        message = [SystemMessage(content=self.SYSTEM_PROMPT), HumanMessage(content=str(question))
                  ]
        llm_response = llm.invoke(message)
        return self.parser.invoke(llm_response)
        

    def classify_dataframe(self, df_final_labels, model_name) -> pd.DataFrame:
        result = pd.DataFrame()
        result['question'] = df_final_labels['question']

        llm = ChatOpenAI(api_key=self.API_KEY, base_url=self.url, model=model_name)
        labels = []
        for question in tqdm(df_final_labels['question'], desc=f'{model_name}'):
            label = self.classify_question(llm, question)
            labels.append(label)
            
        result[model_name] = labels
        return result


In [15]:
models = ['gpt-oss-20b', 'gpt-oss-120b', 'llama-4-scout', 'llama-4-maverick', 'qwen3-14b', 'qwen3-32b', 'qwen3-235b-a22b']
API_KEY = os.getenv('API_KEY')
url = 'https://bothub.chat/api/v2/openai/v1'

classifier = LLM_classifier(models, API_KEY, SYSTEM_PROMPT, url)

In [127]:
classifier = LLM_classifier(models, API_KEY, SYSTEM_PROMPT, url)
results_df_llama_4_scout = classifier.classify_dataframe(df_final_labels.loc[:, ['question']], 'llama-4-scout')

llama-4-scout: 100%|██████████████████████████| 329/329 [13:16<00:00,  2.42s/it]


In [128]:
results_df_llama_4_scout

,question,llama-4-scout
0,"Девушки, кто-нибудь подал на увеличение пособи...",1
1,"А вы первый раз подавали? Я повторно, просто",1
2,Что вы тогда переживаете? У вас справка на рук...,0
3,"Победа, можно личный вопрос на публику? Даже д...",0
4,"Доброе утро,скажите,пожалуйста,а есть у нас в ...",1
...,...,...
324,Вы соблюдаете время бодрствования днём ?),0
325,А для Порчи какой лучше класс?,1
326,"Девочки, Чангу! Кто заберёт шлем?\nразмер S, и...",1
327,А под что маскировать?,0


In [135]:
results_df_llama_4_scout.to_csv('results_df_llama_4_scout.csv', index=False)

In [137]:
results_df_gpt_oss_20b = classifier.classify_dataframe(df_final_labels.loc[:, ['question']], 'gpt-oss-20b')
results_df_gpt_oss_20b

gpt-oss-20b: 100%|████████████████████████████| 329/329 [22:21<00:00,  4.08s/it]


,question,gpt-oss-20b
0,"Девушки, кто-нибудь подал на увеличение пособи...",1
1,"А вы первый раз подавали? Я повторно, просто",1
2,Что вы тогда переживаете? У вас справка на рук...,0
3,"Победа, можно личный вопрос на публику? Даже д...",1
4,"Доброе утро,скажите,пожалуйста,а есть у нас в ...",1
...,...,...
324,Вы соблюдаете время бодрствования днём ?),1
325,А для Порчи какой лучше класс?,виновколы?»\n\n-??...\n\n-раз?год- Всово? ......
326,"Девочки, Чангу! Кто заберёт шлем?\nразмер S, и...",1
327,А под что маскировать?,"фик ves (.aji?weak? chłрюшка? ...бах —,疑остоят..."


In [138]:
results_df_gpt_oss_20b.to_csv('results_df_gpt_oss_20b.csv', index=False)

In [141]:
results_df_gpt_oss_120b = classifier.classify_dataframe(df_final_labels.loc[:, ['question']], 'gpt-oss-120b')
results_df_gpt_oss_120b

gpt-oss-120b: 100%|███████████████████████████| 329/329 [25:48<00:00,  4.71s/it]


,question,gpt-oss-120b
0,"Девушки, кто-нибудь подал на увеличение пособи...",1
1,"А вы первый раз подавали? Я повторно, просто",1
2,Что вы тогда переживаете? У вас справка на рук...,0
3,"Победа, можно личный вопрос на публику? Даже д...",0
4,"Доброе утро,скажите,пожалуйста,а есть у нас в ...",1
...,...,...
324,Вы соблюдаете время бодрствования днём ?),0
325,А для Порчи какой лучше класс?,1
326,"Девочки, Чангу! Кто заберёт шлем?\nразмер S, и...",1
327,А под что маскировать?,1


In [142]:
results_df_gpt_oss_120b.to_csv('results_df_gpt_oss_120b.csv', index=False)

In [145]:
results_df_llama_4_maverick = classifier.classify_dataframe(df_final_labels.loc[:, ['question']], 'llama-4-maverick')
results_df_llama_4_maverick

llama-4-maverick: 100%|███████████████████████| 329/329 [18:22<00:00,  3.35s/it]


,question,llama-4-maverick
0,"Девушки, кто-нибудь подал на увеличение пособи...",1
1,"А вы первый раз подавали? Я повторно, просто",0
2,Что вы тогда переживаете? У вас справка на рук...,0
3,"Победа, можно личный вопрос на публику? Даже д...",0
4,"Доброе утро,скажите,пожалуйста,а есть у нас в ...",1
...,...,...
324,Вы соблюдаете время бодрствования днём ?),0
325,А для Порчи какой лучше класс?,0
326,"Девочки, Чангу! Кто заберёт шлем?\nразмер S, и...",0
327,А под что маскировать?,0


In [146]:
results_df_llama_4_maverick.to_csv('results_df_llama_4_maverick.csv', index=False)

In [15]:
results_df_qwen3_14b = classifier.classify_dataframe(df_final_labels.loc[:, ['question']], 'qwen3-14b')
results_df_qwen3_14b

qwen3-14b: 100%|██████████████████████████████| 329/329 [40:15<00:00,  7.34s/it]


,question,qwen3-14b
0,"Девушки, кто-нибудь подал на увеличение пособи...",1
1,"А вы первый раз подавали? Я повторно, просто",0
2,Что вы тогда переживаете? У вас справка на рук...,0
3,"Победа, можно личный вопрос на публику? Даже д...",0
4,"Доброе утро,скажите,пожалуйста,а есть у нас в ...",1
...,...,...
324,Вы соблюдаете время бодрствования днём ?),0
325,А для Порчи какой лучше класс?,0
326,"Девочки, Чангу! Кто заберёт шлем?\nразмер S, и...",0
327,А под что маскировать?,0


In [19]:
results_df_qwen3_14b.to_csv('results_df_qwen3_14b.csv', index=False)

In [21]:
results_df_qwen3_32b = classifier.classify_dataframe(df_final_labels.loc[:, ['question']], 'qwen3-32b')
results_df_qwen3_32b

qwen3-32b: 100%|████████████████████████████| 329/329 [1:02:20<00:00, 11.37s/it]


,question,qwen3-32b
0,"Девушки, кто-нибудь подал на увеличение пособи...",1
1,"А вы первый раз подавали? Я повторно, просто",1
2,Что вы тогда переживаете? У вас справка на рук...,0
3,"Победа, можно личный вопрос на публику? Даже д...",0
4,"Доброе утро,скажите,пожалуйста,а есть у нас в ...",1
...,...,...
324,Вы соблюдаете время бодрствования днём ?),0
325,А для Порчи какой лучше класс?,1
326,"Девочки, Чангу! Кто заберёт шлем?\nразмер S, и...",0
327,А под что маскировать?,0


In [22]:
results_df_qwen3_32b.to_csv('results_df_qwen3_32b.csv', index=False)

In [28]:
results_df_qwen3_235_a22b = classifier.classify_dataframe(df_final_labels.loc[:, ['question']], 'qwen3-235b-a22b')
results_df_qwen3_235_a22b

qwen3-235b-a22b: 100%|████████████████████████| 329/329 [48:12<00:00,  8.79s/it]


,question,qwen3-235b-a22b
0,"Девушки, кто-нибудь подал на увеличение пособи...",1
1,"А вы первый раз подавали? Я повторно, просто",1
2,Что вы тогда переживаете? У вас справка на рук...,0
3,"Победа, можно личный вопрос на публику? Даже д...",0
4,"Доброе утро,скажите,пожалуйста,а есть у нас в ...",1
...,...,...
324,Вы соблюдаете время бодрствования днём ?),0
325,А для Порчи какой лучше класс?,1
326,"Девочки, Чангу! Кто заберёт шлем?\nразмер S, и...",1
327,А под что маскировать?,0


In [29]:
results_df_qwen3_235_a22b.to_csv('results_df_qwen3_235_a22b.csv', index=False)

In [17]:
results_df_gpt_oss_20b = pd.read_csv('results_df_gpt_oss_20b.csv')
results_df_gpt_oss_120b = pd.read_csv('results_df_gpt_oss_120b.csv')

results_df_llama_4_scout = pd.read_csv('results_df_llama_4_scout.csv')
results_df_llama_4_maverick = pd.read_csv('results_df_llama_4_maverick.csv')

results_df_qwen3_14b = pd.read_csv('results_df_qwen3_14b.csv')
results_df_qwen3_32b = pd.read_csv('results_df_qwen3_32b.csv')
results_df_qwen3_235_a22b = pd.read_csv('results_df_qwen3_235_a22b.csv')

df_result_models_labels = pd.concat([results_df_gpt_oss_20b, results_df_gpt_oss_120b.iloc[:, 1], 
                                     results_df_llama_4_scout.iloc[:, 1], results_df_llama_4_maverick.iloc[:, 1],
                                     results_df_qwen3_14b.iloc[:, 1], results_df_qwen3_32b.iloc[:, 1], results_df_qwen3_235_a22b.iloc[:, 1]], 
                                    axis=1)

In [19]:
df_result_models_labels.head()

,question,gpt-oss-20b,gpt-oss-120b,llama-4-scout,llama-4-maverick,qwen3-14b,qwen3-32b,qwen3-235b-a22b
0,"Девушки, кто-нибудь подал на увеличение пособи...",1,1,1,1.0,1,1.0,1
1,"А вы первый раз подавали? Я повторно, просто",1,1,1,0.0,0,1.0,1
2,Что вы тогда переживаете? У вас справка на рук...,0,0,0,0.0,0,0.0,0
3,"Победа, можно личный вопрос на публику? Даже д...",1,0,0,0.0,0,0.0,0
4,"Доброе утро,скажите,пожалуйста,а есть у нас в ...",1,1,1,1.0,1,1.0,1


In [21]:
df_result_models_labels.to_csv('df_results_models_labels.csv', index=False)

### F1 scores

In [24]:
from sklearn.metrics import f1_score

In [ ]:
# data cleaning
def is_label(x: str):
    temp = str(x).strip()
    if len(temp) == 0: return -1
    if temp[0] == '0' : return 0
    if temp[0] == '1' : return 1
    return -1

for col in df_result_models_labels.columns[1:]:
    df_result_models_labels[col] = (df_result_models_labels[col].apply(is_label).astype('Int64'))


In [26]:
#df_final = pd.read_csv('sampled_questions - agg.csv')

y_true = df_final['final'].astype(int)

models_f1 = {}
for model in models:
    y_pred = df_result_models_labels[model].fillna(-1).astype('Int64')
    models_f1[model] = f1_score(y_true, y_pred, average='macro');

In [27]:
for model, score in models_f1.items():
    print(f'{model}: {score}\n')

gpt-oss-20b: 0.32547016763804604

gpt-oss-120b: 0.6438843358589763

llama-4-scout: 0.47214654261938876

llama-4-maverick: 0.5207612456747405

qwen3-14b: 0.7641937672613985

qwen3-32b: 0.5117012480049228

qwen3-235b-a22b: 0.7583687669114805

